# CALE Experiment Visualization

This notebook visualizes the JSON report produced by `experiment.py`. It now supports both:

1. **supervised settings** with response-level expert labels, where `accuracy` and `macro_f1` are available;
2. **FEVER-style claim datasets** without response-level expert labels, where evaluator comparison should focus on prediction distributions, construct subscores, and stress robustness.


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

plt.style.use("default")


## 1. Load Results

Set `RESULTS_PATH` to the JSON report generated by `experiment.py`.
Do not point it at the intermediate `*.jsonl` file from `generate_responses.py`.


In [ ]:
RESULTS_PATH = Path("outputs/fever_dev_qwen25_15b_llama32_1b_neutral_smoke_eval_report.json")
OUTDIR = Path("figures") / RESULTS_PATH.stem
OUTDIR.mkdir(parents=True, exist_ok=True)

raw_text = RESULTS_PATH.read_text(encoding="utf-8")
nonempty_lines = [line for line in raw_text.splitlines() if line.strip()]
if len(nonempty_lines) > 1 and all(line.lstrip().startswith("{") for line in nonempty_lines[:2]):
    raise ValueError(
        "RESULTS_PATH points to a JSONL-style predictions file, not the report JSON from experiment.py. "
        "Run experiment.py first and use a file like outputs/fever_dev_qwen25_15b_llama32_1b_neutral_smoke_eval_report.json instead."
    )

try:
    report = json.loads(raw_text)
except json.JSONDecodeError as exc:
    raise ValueError(
        f"Could not parse {RESULTS_PATH} as the JSON report written by experiment.py: {exc}"
    ) from exc

if isinstance(report, list):
    raise ValueError(
        "RESULTS_PATH points to a JSON list, not the report JSON from experiment.py. "
        "Use a file like outputs/fever_dev_qwen25_15b_llama32_1b_neutral_smoke_eval_report.json instead."
    )

required_keys = {"metrics"}
missing_keys = required_keys - set(report.keys())
if missing_keys:
    raise ValueError(
        f"Report JSON is missing required keys: {sorted(missing_keys)}. "
        "Expected the JSON output written by experiment.py."
    )

if "predictions" not in report:
    print("This looks like a --summary-only report: row-level prediction plots will be skipped.")
if "stress_tests" not in report and "stress_summary" in report:
    print("Stress summary is available, but perturbation-level plots require a non-summary report.")

report.keys()


## 2. Metrics Table

If you are using FEVER without response-level expert labels, `accuracy`, `macro_f1`, and `checklist_f1` may be null. In that case, the later sections on prediction distributions and stress robustness become more important.


In [ ]:
metrics_df = (
    pd.DataFrame(report["metrics"])
    .T
    .reset_index()
    .rename(columns={"index": "variant"})
)
metrics_df


## 3. Plot Only Available Metrics

This cell skips metrics that are entirely null.


In [ ]:
def plot_metric(metric_name, ylim=(0, 1)):
    if metric_name not in metrics_df.columns:
        print(f"Missing metric: {metric_name}")
        return
    df = metrics_df[["variant", metric_name]].dropna()
    if df.empty:
        print(f"Skipping {metric_name}: no non-null values.")
        return

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(df["variant"], df[metric_name])
    ax.set_title(metric_name)
    ax.set_ylabel(metric_name)
    ax.set_ylim(*ylim)
    ax.tick_params(axis="x", rotation=30)
    for label in ax.get_xticklabels():
        label.set_ha("right")
    fig.tight_layout()
    fig.savefig(OUTDIR / f"metric_{metric_name}.png", dpi=200)
    plt.show()

for metric in [
    "accuracy",
    "macro_f1",
    "checklist_f1",
    "mean_uncertainty",
    "misinformation_detection_rate",
    "framing_resistance_rate",
    "source_faithfulness_rate",
    "overclaim_rate_on_nei",
]:
    plot_metric(metric, ylim=(0, 1))


## 4. Grouped Metrics by Evaluation Resource

These views connect the visual analysis to the paper's dataset structure: internal constructed evaluation, domain-transfer resources, and adversarial/resource-risk slices.


In [ ]:
GROUPED_KEYS = {
    "metrics_by_model": "model_name",
    "metrics_by_setting": "evaluation_setting",
    "metrics_by_dataset": "dataset",
    "metrics_by_dataset_role": "dataset_role",
    "metrics_by_domain": "domain",
    "metrics_by_risk_level": "risk_level",
}

def flatten_grouped_metrics(grouped_key):
    rows = []
    for group_name, variant_metrics in report.get(grouped_key, {}).items():
        for variant, values in variant_metrics.items():
            row = {"group": group_name, "variant": variant}
            row.update(values)
            rows.append(row)
    return pd.DataFrame(rows)

grouped_metric_tables = {key: flatten_grouped_metrics(key) for key in GROUPED_KEYS}
for key, df in grouped_metric_tables.items():
    print(key, df.shape)


In [ ]:
GROUPED_PLOT_METRICS = [
    "misinformation_detection_rate",
    "framing_resistance_rate",
    "source_faithfulness_rate",
    "overclaim_rate_on_nei",
]

for grouped_key, group_label in GROUPED_KEYS.items():
    df = grouped_metric_tables[grouped_key]
    if df.empty:
        print(f"No grouped metrics for {group_label}.")
        continue
    for metric in GROUPED_PLOT_METRICS:
        if metric not in df.columns or df[metric].dropna().empty:
            continue
        pivot = df.pivot_table(index="variant", columns="group", values=metric, aggfunc="mean")
        fig, ax = plt.subplots(figsize=(max(8, 1.7 * len(pivot.columns)), max(4, 0.45 * len(pivot))))
        image = ax.imshow(pivot.fillna(0).values, aspect="auto", vmin=0, vmax=1)
        fig.colorbar(image, ax=ax, label=metric)
        ax.set_xticks(range(len(pivot.columns)))
        ax.set_xticklabels(pivot.columns, rotation=30, ha="right")
        ax.set_yticks(range(len(pivot.index)))
        ax.set_yticklabels(pivot.index)
        ax.set_title(f"{metric} by {group_label}")
        fig.tight_layout()
        fig.savefig(OUTDIR / f"{grouped_key}_{metric}.png", dpi=200)
        plt.show()


## 5. Prediction Table

This table is the most useful view when gold response labels are not available.


In [ ]:
pred_df = pd.DataFrame(report.get("predictions", []))
pred_df.head()


## 6. Prediction Label Distribution by Variant

This shows how often each evaluator variant predicts `corrected`, `partially_corrected`, `uncertain`, or `not_corrected`.


In [ ]:
if not pred_df.empty:
    label_counts = (
        pred_df.groupby(["variant", "label"]).size()
        .reset_index(name="count")
    )
    label_counts
else:
    print("No predictions found.")


In [ ]:
if not pred_df.empty:
    pivot = label_counts.pivot(index="variant", columns="label", values="count").fillna(0)
    pivot = pivot.div(pivot.sum(axis=1), axis=0)

    ax = pivot.plot(kind="bar", stacked=True, figsize=(10, 5))
    ax.set_title("Prediction label distribution by evaluator variant")
    ax.set_ylabel("proportion")
    ax.tick_params(axis="x", rotation=30)
    for label in ax.get_xticklabels():
        label.set_ha("right")
    plt.tight_layout()
    plt.savefig(OUTDIR / "prediction_label_distribution.png", dpi=200)
    plt.show()


## 7. Score Distribution by Variant

This helps compare whether CALE and direct judges produce systematically different score ranges.


In [ ]:
if not pred_df.empty and "score" in pred_df.columns:
    fig, ax = plt.subplots(figsize=(10, 5))
    pred_df.boxplot(column="score", by="variant", ax=ax, grid=False, rot=30)
    ax.set_title("Score distribution by evaluator variant")
    ax.set_ylabel("score")
    plt.suptitle("")
    plt.tight_layout()
    plt.savefig(OUTDIR / "score_distribution_by_variant.png", dpi=200)
    plt.show()


## 8. Stress Summary

These plots compare robustness under perturbations. Lower invariance score shift and lower label-change rate are better for construct-irrelevant perturbations.


In [ ]:
if "stress_summary" in report:
    stress_summary_df = (
        pd.DataFrame(report["stress_summary"])
        .T
        .reset_index()
        .rename(columns={"index": "variant"})
    )
else:
    stress_summary_df = pd.DataFrame()

stress_summary_df


In [ ]:
def plot_stress_metric(metric_name):
    if stress_summary_df.empty or metric_name not in stress_summary_df.columns:
        print(f"Missing stress metric: {metric_name}")
        return
    df = stress_summary_df[["variant", metric_name]].dropna()
    if df.empty:
        print(f"No values for stress metric: {metric_name}")
        return

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(df["variant"], df[metric_name])
    ax.set_title(metric_name)
    ax.set_ylabel(metric_name)
    ax.tick_params(axis="x", rotation=30)
    for label in ax.get_xticklabels():
        label.set_ha("right")
    fig.tight_layout()
    fig.savefig(OUTDIR / f"stress_{metric_name}.png", dpi=200)
    plt.show()

for metric in [
    "invariance_label_change_rate",
    "mean_abs_invariance_score_shift",
    "mean_sensitivity_score_drop",
]:
    plot_stress_metric(metric)


## 9. Stress Summary by Model

When `experiment.py` is run with multiple target models, this summary-level view compares robustness without requiring raw `stress_tests` rows.


In [ ]:
def flatten_nested_summary(summary_key):
    rows = []
    for group_name, variant_metrics in report.get(summary_key, {}).items():
        for variant, values in variant_metrics.items():
            row = {"group": group_name, "variant": variant}
            row.update(values)
            rows.append(row)
    return pd.DataFrame(rows)

stress_by_model_df = flatten_nested_summary("stress_summary_by_model")
stress_by_model_df.head() if not stress_by_model_df.empty else "No stress_summary_by_model found."


In [ ]:
if not stress_by_model_df.empty:
    for metric in ["invariance_label_change_rate", "mean_abs_invariance_score_shift", "mean_sensitivity_score_drop", "cross_framing_label_flip_rate"]:
        if metric not in stress_by_model_df.columns or stress_by_model_df[metric].dropna().empty:
            continue
        pivot = stress_by_model_df.pivot_table(index="variant", columns="group", values=metric, aggfunc="mean")
        fig, ax = plt.subplots(figsize=(max(8, 1.7 * len(pivot.columns)), max(4, 0.45 * len(pivot))))
        image = ax.imshow(pivot.fillna(0).values, aspect="auto")
        fig.colorbar(image, ax=ax, label=metric)
        ax.set_xticks(range(len(pivot.columns)))
        ax.set_xticklabels(pivot.columns, rotation=30, ha="right")
        ax.set_yticks(range(len(pivot.index)))
        ax.set_yticklabels(pivot.index)
        ax.set_title(f"{metric} by model")
        fig.tight_layout()
        fig.savefig(OUTDIR / f"stress_by_model_{metric}.png", dpi=200)
        plt.show()
else:
    print("No summary-level stress-by-model data available.")


## 10. Score Shift by Perturbation

For construct-irrelevant perturbations, score shifts should be close to zero. Unsupported extra claims should usually produce a score drop.


In [ ]:
stress_df = pd.DataFrame(report.get("stress_tests", []))
stress_df.head()


In [ ]:
if not stress_df.empty:
    for variant in stress_df["variant"].unique():
        sub = stress_df[stress_df["variant"] == variant]
        means = sub.groupby("perturbation")["score_shift"].mean().sort_values()

        fig, ax = plt.subplots(figsize=(10, 4))
        ax.bar(means.index, means.values)
        ax.axhline(0, color="black", linewidth=1)
        ax.set_title(f"Mean score shift by perturbation: {variant}")
        ax.set_ylabel("score_shift")
        ax.tick_params(axis="x", rotation=30)
        for label in ax.get_xticklabels():
            label.set_ha("right")
        fig.tight_layout()
        fig.savefig(OUTDIR / f"score_shift_{variant}.png", dpi=200)
        plt.show()
else:
    print("No stress_tests found. Run experiment.py with --stress.")


## 11. CALE Subscore Heatmap

This is one of the most important plots for CALE, because it shows construct-level interpretability rather than just a final label.


In [ ]:
prediction_rows = []
for pred in report.get("predictions", []):
    if pred.get("variant") != "full_attack_aware_cale":
        continue
    for dimension, score in pred.get("subscores", {}).items():
        prediction_rows.append({
            "id": pred["id"],
            "dimension": dimension,
            "score": score,
        })

subscore_df = pd.DataFrame(prediction_rows)
subscore_df.head()


In [ ]:
if not subscore_df.empty:
    heatmap_table = subscore_df.pivot(index="id", columns="dimension", values="score")

    fig, ax = plt.subplots(figsize=(11, max(4, 0.35 * len(heatmap_table))))
    image = ax.imshow(heatmap_table.values, aspect="auto", vmin=0, vmax=1)
    fig.colorbar(image, ax=ax, label="subscore")
    ax.set_xticks(range(len(heatmap_table.columns)))
    ax.set_xticklabels(heatmap_table.columns, rotation=30, ha="right")
    ax.set_yticks(range(len(heatmap_table.index)))
    ax.set_yticklabels(heatmap_table.index)
    ax.set_title("CALE dimension subscores")
    fig.tight_layout()
    fig.savefig(OUTDIR / "cale_subscore_heatmap.png", dpi=200)
    plt.show()
else:
    print("No full_attack_aware_cale subscores found.")


## 12. FEVER-Specific Optional View

If your generated dataset still contains `reference_label` from FEVER, this plot shows how evaluator outputs distribute across FEVER claim labels.


In [ ]:
if not pred_df.empty and "reference_label" in pred_df.columns:
    fever_view = pred_df[["variant", "reference_label", "label"]].dropna().copy()
else:
    fever_view = pd.DataFrame()

if not fever_view.empty:
    chosen_variant = "full_attack_aware_cale" if "full_attack_aware_cale" in fever_view["variant"].unique() else fever_view["variant"].iloc[0]
    sub = fever_view[fever_view["variant"] == chosen_variant]
    pivot = (
        sub.groupby(["reference_label", "label"]).size()
        .reset_index(name="count")
        .pivot(index="reference_label", columns="label", values="count")
        .fillna(0)
    )
    pivot = pivot.div(pivot.sum(axis=1), axis=0)
    ax = pivot.plot(kind="bar", stacked=True, figsize=(9, 5))
    ax.set_title(f"Evaluator label distribution by FEVER label ({chosen_variant})")
    ax.set_ylabel("proportion")
    ax.tick_params(axis="x", rotation=20)
    for label in ax.get_xticklabels():
        label.set_ha("right")
    plt.tight_layout()
    plt.savefig(OUTDIR / "fever_reference_label_distribution.png", dpi=200)
    plt.show()
else:
    print("No reference_label found in predictions.")


## 13. Export Tables

Save metric and stress tables as CSV for paper tables.


In [ ]:
metrics_df.to_csv(OUTDIR / "metrics_table.csv", index=False)
if not stress_summary_df.empty:
    stress_summary_df.to_csv(OUTDIR / "stress_summary_table.csv", index=False)
if not subscore_df.empty:
    subscore_df.to_csv(OUTDIR / "cale_subscores_long.csv", index=False)
if not pred_df.empty:
    pred_df.to_csv(OUTDIR / "predictions_table.csv", index=False)

print(f"Saved figures and tables to: {OUTDIR.resolve()}")


## 14. Attack-Aware Dimension Plots

These plots focus on the new attack-aware CALE dimensions.

In [ ]:
if not pred_df.empty:
    attack_dims = ["Misinformation Detection", "Framing Resistance", "Source Faithfulness"]
    rows = []
    for _, row in pred_df.iterrows():
        for dim in attack_dims:
            if isinstance(row.get("subscores"), dict) and dim in row["subscores"]:
                rows.append({"variant": row["variant"], "dimension": dim, "score": row["subscores"][dim]})
    attack_df = pd.DataFrame(rows)
    attack_df.head()
else:
    print("No predictions found.")


In [ ]:
if "attack_df" in globals() and not attack_df.empty:
    summary = attack_df.groupby(["variant", "dimension"]) ["score"].mean().reset_index()
    for dim in summary["dimension"].unique():
        sub = summary[summary["dimension"] == dim]
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.bar(sub["variant"], sub["score"])
        ax.set_title(dim)
        ax.set_ylim(0, 1)
        ax.tick_params(axis="x", rotation=30)
        for label in ax.get_xticklabels():
            label.set_ha("right")
        fig.tight_layout()
        fig.savefig(OUTDIR / f"attack_dim_{dim.replace(' ', '_')}.png", dpi=200)
        plt.show()
else:
    print("No attack-aware subscores available.")


## 15. Model-Level Comparison

When `model_name` is available, these plots compare target models across the same evaluator variants.


In [ ]:
if not pred_df.empty and 'model_name' in pred_df.columns:
    model_variant_scores = (
        pred_df.groupby(['model_name', 'variant'])['score']
        .mean()
        .reset_index()
    )
    model_variant_scores.head()
else:
    model_variant_scores = pd.DataFrame()
    print('No model_name field found in predictions.')


In [ ]:
if not model_variant_scores.empty:
    pivot = model_variant_scores.pivot(index='model_name', columns='variant', values='score')
    fig, ax = plt.subplots(figsize=(11, max(4, 0.6 * len(pivot))))
    image = ax.imshow(pivot.values, aspect='auto', vmin=0, vmax=1)
    fig.colorbar(image, ax=ax, label='mean score')
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=30, ha='right')
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_title('Mean evaluator score by model and variant')
    fig.tight_layout()
    fig.savefig(OUTDIR / 'model_variant_score_heatmap.png', dpi=200)
    plt.show()
else:
    print('No model-level scores available.')


## 16. Framing Comparison

These plots compare how predictions shift across `neutral`, `assertive`, `authoritative`, and `polite_misleading` framings.


In [ ]:
if not pred_df.empty and 'framing_style' in pred_df.columns:
    framing_scores = (
        pred_df.groupby(['variant', 'framing_style'])['score']
        .mean()
        .reset_index()
    )
    framing_scores.head()
else:
    framing_scores = pd.DataFrame()
    print('No framing_style field found in predictions.')


In [ ]:
if not framing_scores.empty:
    pivot = framing_scores.pivot(index='variant', columns='framing_style', values='score').fillna(0)
    ax = pivot.plot(kind='bar', figsize=(10, 5))
    ax.set_title('Mean score by evaluator variant and framing style')
    ax.set_ylabel('mean score')
    ax.tick_params(axis='x', rotation=30)
    for label in ax.get_xticklabels():
        label.set_ha('right')
    plt.tight_layout()
    plt.savefig(OUTDIR / 'variant_by_framing_score.png', dpi=200)
    plt.show()
else:
    print('No framing comparison available.')


## 17. Model-Level Label Distribution

This stacked view helps compare how different target models are judged under the same evaluator variant.


In [ ]:
if not pred_df.empty and {'model_name', 'variant', 'label'}.issubset(pred_df.columns):
    chosen_variant = 'full_attack_aware_cale' if 'full_attack_aware_cale' in pred_df['variant'].unique() else pred_df['variant'].iloc[0]
    sub = pred_df[pred_df['variant'] == chosen_variant]
    model_label = sub.groupby(['model_name', 'label']).size().reset_index(name='count')
    pivot = model_label.pivot(index='model_name', columns='label', values='count').fillna(0)
    pivot = pivot.div(pivot.sum(axis=1), axis=0)
    ax = pivot.plot(kind='bar', stacked=True, figsize=(11, 5))
    ax.set_title(f'Prediction label distribution by model ({chosen_variant})')
    ax.set_ylabel('proportion')
    ax.tick_params(axis='x', rotation=30)
    for label in ax.get_xticklabels():
        label.set_ha('right')
    plt.tight_layout()
    plt.savefig(OUTDIR / 'model_label_distribution.png', dpi=200)
    plt.show()
else:
    print('No model-level label distribution available.')


## 18. Stress Tests by Model

This view is useful when you run the same perturbations for multiple target models.


In [ ]:
if not stress_df.empty and {'model_name', 'variant'}.issubset(stress_df.columns):
    chosen_variant = 'full_attack_aware_cale' if 'full_attack_aware_cale' in stress_df['variant'].unique() else stress_df['variant'].iloc[0]
    sub = stress_df[(stress_df['variant'] == chosen_variant) & (stress_df['perturbation'].isin(['neutral_falsehood', 'assertive_falsehood', 'authoritative_falsehood']))]
    if not sub.empty:
        flip_df = sub.groupby(['model_name', 'perturbation'])['label_changed'].mean().reset_index()
        pivot = flip_df.pivot(index='model_name', columns='perturbation', values='label_changed').fillna(0)
        fig, ax = plt.subplots(figsize=(11, max(4, 0.6 * len(pivot))))
        image = ax.imshow(pivot.values, aspect='auto', vmin=0, vmax=1)
        fig.colorbar(image, ax=ax, label='label flip rate')
        ax.set_xticks(range(len(pivot.columns)))
        ax.set_xticklabels(pivot.columns, rotation=30, ha='right')
        ax.set_yticks(range(len(pivot.index)))
        ax.set_yticklabels(pivot.index)
        ax.set_title(f'Cross-framing label flip rate by model ({chosen_variant})')
        fig.tight_layout()
        fig.savefig(OUTDIR / 'model_cross_framing_flip_heatmap.png', dpi=200)
        plt.show()
    else:
        print('No model-level adversarial framing stress data available.')
else:
    print('No model_name field found in stress tests.')
